# Generate Project Documentation
ITDS 2373 — Anime Acquisition Strategy

Run after exporting `final_anime_dataset.csv` from the main notebook. This notebook creates two Word files without requiring raw reviews or existing Word documents. Historical cleaning counts are taken from the executed capstone notebook and must be updated if the source data changes.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

OUTPUT_DIR = Path("Anime_Project_Output_Final")
df = pd.read_csv(OUTPUT_DIR / "final_anime_dataset.csv")
assert df.shape[1] == 79 and df.uid.is_unique and df.uid.notna().all()
components = ["Overall", "Story", "Animation", "Sound", "Character", "Enjoyment"]


## Recalculate data quality and dictionary
CSV does not preserve pandas nullable integer types; the dictionary reports types as loaded from CSV. Missing counts include the same placeholder rules as the main notebook.

In [4]:
missing_counts = df.isna().sum()
placeholders = ["", "not available", "unknown", "n/a", "none", "null"]
for col in df.select_dtypes(include=["object", "string"]):
    if col != "content_category":
        missing_counts[col] += df[col].astype("string").str.strip().str.lower().isin(placeholders).sum()

completeness_report = pd.DataFrame({"Missing Count": missing_counts,
    "Missing %": (missing_counts / len(df) * 100).round(2)})
completeness_report["Completeness %"] = 100 - completeness_report["Missing %"]
completeness_report.index.name = "Column"

genre_cols = [col for col in df if col.startswith("genre_") and col != "genre_count"]
avg_cols = [f"avg_review_{name}" for name in components]
count_cols = [f"count_review_{name}" for name in components]
validity = {}
for col, low, high in [("score", 0, 10), ("popularity_index", 0, 100),
                       ("avg_review_score", 1, 10)] + [(col, 1, 10) for col in avg_cols]:
    validity[col + " outside range"] = (df[col].notna() & ~df[col].between(low, high)).sum()
for col in ["episodes", "members", "review_count", "favorites_count", "anime_age"]:
    validity[col + " negative"] = df[col].lt(0).sum()
validity["Invalid popularity rank"] = df["popularity"].lt(1).sum()
for col in count_cols:
    validity[col + " invalid"] = (df[col].isna() | df[col].lt(0) | df[col].mod(1).ne(0)).sum()
flags = genre_cols + ["adult_content_flag", "suggestive_content_flag", "franchise_indicator"]
validity["Invalid binary flags"] = (~df[flags].isin([0, 1])).sum().sum()
categories = ["Adult Content", "Suggestive Content", "Unknown", "No Adult Genre Identified"]
validity["Invalid content category"] = (~df.content_category.isin(categories)).sum()
validity_report = pd.Series(validity, name="Invalid Records").rename_axis("Validation Check").reset_index()

known = df.release_year.notna()
consistency = {
    "Genre count mismatch": df[genre_cols].sum(axis=1).ne(df.genre_count).sum(),
    "Release decade mismatch": (known & (df.release_decade.isna() | df.release_decade.ne(df.release_year // 10 * 10))).sum(),
    "Anime age mismatch": (known & (df.anime_age.isna() | df.anime_age.ne(2026 - df.release_year))).sum(),
    "Derived dates without release year": (~known & (df.anime_age.notna() | df.release_decade.notna())).sum(),
    "Review score without reviews": (df.review_count.eq(0) & df.avg_review_score.notna()).sum()
}
for avg, count in zip(avg_cols, count_cols):
    consistency[avg + " / count mismatch"] = ((df[count].eq(0) & df[avg].notna()) |
        (df[count].gt(0) & df[avg].isna()) | df[count].gt(df.review_count)).sum()
consistency_report = pd.Series(consistency, name="Count").rename_axis("Check").reset_index()
review_coverage = pd.DataFrame({"Component": components,
    "Anime With Ratings": df[avg_cols].notna().sum().values,
    "Missing Ratings": df[avg_cols].isna().sum().values,
    "Mean Score": df[avg_cols].mean().round(2).values,
    "Total Valid Ratings": df[count_cols].sum().values})
review_coverage["Coverage %"] = (review_coverage["Anime With Ratings"] / len(df) * 100).round(2)
quality_summary = pd.Series({
    "Total records": len(df), "Total columns": df.shape[1],
    "Overall completeness (%)": round(100 * (1 - missing_counts.sum() / df.size), 2),
    "Duplicate rows": df.duplicated().sum(), "Duplicate anime IDs": df.uid.duplicated().sum(),
    "Missing anime IDs": df.uid.isna().sum(),
    "Invalid numeric/encoded values": validity_report["Invalid Records"].sum(),
    "Consistency issues": consistency_report["Count"].sum(),
    "Binary genre columns": len(genre_cols)
}, name="Result").rename_axis("Metric").reset_index()
display(quality_summary)

,Metric,Result
0,Total records,16216.00
1,Total columns,79.00
2,Overall completeness (%),95.14
3,Duplicate rows,0.00
4,Duplicate anime IDs,0.00
5,Missing anime IDs,0.00
6,Invalid numeric/encoded values,0.00
7,Consistency issues,0.00
8,Binary genre columns,43.00


In [5]:
definitions = {
    'uid': 'Unique anime identifier',
    'title': 'Anime title',
    'synopsis': 'Anime description',
    'genre': 'Anime genre labels',
    'aired': 'Release or broadcast information',
    'episodes': 'Number of episodes',
    'members': 'Number of anime community members',
    'popularity': 'Popularity ranking; lower is better',
    'ranked': 'Overall anime ranking; lower is better',
    'score': 'Overall anime rating',
    'img_url': 'Anime image URL',
    'link': 'Anime information URL',
    'popularity_index': 'Normalized popularity score (0-100)',
    'genre_count': 'Number of unique genres assigned to each anime',
    'franchise_indicator': 'Possible related-release title flag',
    'review_count': 'Number of reviews for each anime',
    'adult_content_flag': '1 if genre text contains Hentai or Erotica; otherwise 0.',
    'suggestive_content_flag': '1 if genre text contains Ecchi; otherwise 0.',
    'content_category': 'Genre-based category; adult takes precedence over suggestive, then unknown for blank genre lists. Not an age rating.',
    'favorites_count': 'Number of profiles favoriting the anime',
    'release_year': 'First release year extracted from the aired column',
    'anime_age': 'Years since first release, calculated as of 2026',
    'release_decade': 'Decade of first release, used for historical trend analysis',
    'avg_review_score': 'Mean valid overall review score (1–10); missing if none.',
}
for col in genre_cols:
    definitions[col] = f"1 if tagged {col[6:]}; otherwise 0."
for name in components:
    definitions[f"avg_review_{name}"] = f"Mean valid {name} rating (1–10); missing if none."
    definitions[f"count_review_{name}"] = f"Number of valid {name} ratings; 0 if none."

assert set(df.columns) == set(definitions), "Dictionary does not match dataset schema"
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Description": df.columns.map(definitions),
    "Missing Values": df.isna().sum().values,
    "Missing Including Placeholders": missing_counts.values
})
display(data_dictionary.head())


,Column,Data Type,Description,Missing Values,Missing Including Placeholders
0,uid,int64,Unique anime identifier,0,0
1,title,object,Anime title,0,0
2,synopsis,object,Anime description,763,770
3,genre,object,Anime genre labels,0,0
4,aired,object,Release or broadcast information,0,273


## Cleaning history
These counts are recorded from the saved execution of the capstone notebook, not inferred from the final CSV. 

In [6]:
cleaning_history = pd.DataFrame([
    ["Anime", 19311, 2943, 152, 16216],
    ["Profiles", 81727, 33825, 17, 47885],
    ["Reviews", 192112, 61593, 0, 130519]
], columns=["Dataset", "Original", "Exact duplicates", "Repeated IDs", "Cleaned"])
invalid_overall = 2
component_cleaning = pd.DataFrame({"Component": components,
    "Zero ratings": [1,5910,5981,6283,6058,6350],
    "Other invalid": [1]*6})
assert len(df) == cleaning_history.loc[0, "Cleaned"], "Update cleaning history for this dataset"
content_summary = df.content_category.value_counts().rename_axis("Category").reset_index(name="Titles")


## Word formatting helpers

In [7]:
def new_document(title):
    doc = Document()
    sec = doc.sections[0]
    sec.top_margin = sec.bottom_margin = Inches(.7)
    sec.left_margin = sec.right_margin = Inches(.7)
    normal = doc.styles["Normal"]
    normal.font.name = "Calibri"
    normal.font.size = Pt(11)
    for name in ["Title", "Heading 1", "Heading 2"]:
        doc.styles[name].font.color.rgb = RGBColor.from_string("17365D")
    doc.add_heading(title, 0)
    doc.add_paragraph("Anime Acquisition Strategy Capstone | Tobechukwu")
    footer = sec.footer.paragraphs[0]
    footer.alignment = WD_ALIGN_PARAGRAPH.CENTER
    footer.add_run(title + " · page ")
    field = OxmlElement("w:fldSimple")
    field.set(qn("w:instr"), "PAGE")
    footer._p.append(field)
    return doc


def add_table(doc, frame, widths, description_cols=()):
    table = doc.add_table(rows=1, cols=len(frame.columns))
    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    sec = doc.sections[0]
    available = sec.page_width - sec.left_margin - sec.right_margin
    sizes = [int(available*w/sum(widths)) for w in widths]
    for col, size in zip(table.columns, sizes):
        col.width = size
    for i, name in enumerate(frame.columns):
        table.rows[0].cells[i].text = str(name)
    for values in frame.itertuples(index=False, name=None):
        for cell, value in zip(table.add_row().cells, values):
            cell.text = str(value)
    header = OxmlElement("w:tblHeader")
    table.rows[0]._tr.get_or_add_trPr().append(header)
    borders = OxmlElement("w:tblBorders")
    for edge in ["top", "left", "bottom", "right", "insideH", "insideV"]:
        el = OxmlElement("w:" + edge)
        for k,v in {"val":"single", "sz":"4", "color":"A7B9CC"}.items():
            el.set(qn("w:"+k), v)
        borders.append(el)
    table._tbl.tblPr.append(borders)
    for ri, row in enumerate(table.rows):
        for ci, cell in enumerate(row.cells):
            cell.width = sizes[ci]
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if ri == 0:
                shade = OxmlElement("w:shd")
                shade.set(qn("w:fill"), "DCE8F4")
                cell._tc.get_or_add_tcPr().append(shade)
            for p in cell.paragraphs:
                p.paragraph_format.space_after = Pt(3)
                p.paragraph_format.space_before = Pt(3)
                p.alignment = (WD_ALIGN_PARAGRAPH.CENTER if ri == 0 else
                    WD_ALIGN_PARAGRAPH.JUSTIFY if ci in description_cols else
                    WD_ALIGN_PARAGRAPH.RIGHT if pd.api.types.is_numeric_dtype(frame.iloc[:,ci]) else WD_ALIGN_PARAGRAPH.LEFT)
                for run in p.runs:
                    run.font.size = Pt(10)
                    run.bold = ri == 0
    return table


## Generate the data dictionary

In [8]:
doc = new_document("Data Dictionary")
doc.add_paragraph(f"File: final_anime_dataset.csv | {len(df):,} titles | {df.shape[1]} columns | Primary key: uid")
doc.add_paragraph("Missing counts include recognized text placeholders. Unknown is a valid content category. Popularity index increases with popularity; raw rank runs in the opposite direction. Review averages remain missing when no valid ratings exist.")
groups = {
    "Original anime attributes": list(df.columns[:12]),
    "Engineered metrics": [c for c in df if c not in list(df.columns[:12])+genre_cols+avg_cols+count_cols+["adult_content_flag","suggestive_content_flag","content_category"]],
    "Genre indicators": genre_cols,
    "Review component averages": avg_cols,
    "Valid component rating counts": count_cols,
    "Content suitability": ["adult_content_flag","suggestive_content_flag","content_category"]
}
assert sum(map(len,groups.values())) == len(df.columns)
for title, cols in groups.items():
    doc.add_heading(f"{title} ({len(cols)})", 1)
    part = data_dictionary.set_index("Column").loc[cols].reset_index()
    part = part[["Column","Data Type","Missing Including Placeholders","Description"]]
    part.columns = ["Column","Type","Missing","Description"]
    add_table(doc, part, [34,12,12,42], (3,))
doc.save(OUTPUT_DIR / "Anime_Data_Dictionary.docx")


## Generate the engineering report

In [9]:
doc = new_document("Data Cleaning and Engineering Report")
doc.add_paragraph(f"The final dataset contains {len(df):,} anime titles and {df.shape[1]} columns. It supports audience, review and licensing analysis at anime level.")
doc.add_heading("1. Sources and relationships", 1)
doc.add_paragraph("The Kaggle MyAnimeList data contains anime entries, user profiles and reviews. Anime uid links to review anime_uid and to IDs in profile favorites lists. Review and favorite summaries are merged into the anime table using left joins to retain all titles.")
doc.add_heading("2. Cleaning and preparation", 1)
add_table(doc, cleaning_history, [20,20,20,20,20])
doc.add_paragraph(f"Exact duplicates were removed first, followed by repeated keys, retaining the first occurrence. Numeric fields were converted with invalid conversions treated as missing. {invalid_overall} out-of-range overall review scores were set to missing; component ratings outside 1–10 were excluded from averages.")
add_table(doc, component_cleaning, [40,30,30])
doc.add_heading("3. Text preprocessing", 1)
doc.add_paragraph("Review text was HTML-decoded and stripped of rating headers, HTML tags, URLs, punctuation and numbers. Text was lowercased and English stopwords removed, preserving no, not, nor and never. The processed_text field is retained in reviews_clean.csv for downstream sentiment analysis. This is not a sentiment model.")
doc.add_heading("4. Feature engineering", 1)
features = pd.DataFrame([
    ["Popularity", "Reverse min-max normalization of rank to 0–100."],
    ["Genres", "Count unique tags and encode 43 binary genre indicators."],
    ["Franchise", "Title keyword heuristic for seasons, parts and sequels; not official franchise grouping."],
    ["Release", "Extract first 19xx or 20xx year, calculate age relative to 2026 and decade."],
    ["Favorites", "Count profile favorites matched to anime IDs."],
    ["Reviews", "Aggregate review counts, valid overall mean, six component means and six valid counts."]
], columns=["Feature group","Method"])
add_table(doc, features, [27,73], (1,))
doc.add_heading("5. Content suitability", 1)
doc.add_paragraph("Hentai or Erotica triggers the adult flag; Ecchi triggers the suggestive flag. Adult takes precedence over suggestive, then unknown for empty genre lists. These are genre-based screening indicators, not official age ratings.")
add_table(doc, content_summary, [75,25])
doc.add_heading("6. Data quality and coverage", 1)
add_table(doc, quality_summary, [75,25])
add_table(doc, review_coverage[["Component","Anime With Ratings","Total Valid Ratings","Coverage %"]], [25,25,25,25])
doc.add_heading("7. Reliability and handover", 1)
reviewed = int(df.review_count.gt(0).sum())
doc.add_paragraph(f"{reviewed:,} titles ({reviewed/len(df):.1%}) have reviews. Use review counts alongside averages; zero reviews do not imply poor quality. Missing scores should not be treated as zero. Favorites reflect the supplied profile sample, and franchise flags are approximate. Release coverage ends in {int(df.release_year.max())}; the dataset does not represent current market conditions. Apply content screening at licensing selection and independently verify suitability.")
doc.add_paragraph("Cleaning history is from the saved capstone execution; quality and coverage statistics are recalculated from the exported dataset. Raw reviews are not reprocessed by this documentation notebook.")
doc.save(OUTPUT_DIR / "Anime_Data_Engineering_Report.docx")
print("Created:", OUTPUT_DIR / "Anime_Data_Dictionary.docx", "and", OUTPUT_DIR / "Anime_Data_Engineering_Report.docx")


Created: Anime_Project_Output_Final\Anime_Data_Dictionary.docx and Anime_Project_Output_Final\Anime_Data_Engineering_Report.docx
